In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.metrics import accuracy_score

In [2]:
# 设置随机种子保证可复现
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ==================== 1. 数据加载 ====================
print("加载数据...")
train_df = pd.read_csv(r'.\三级实操题-202603-加密\telecom_churn_train.csv')
test_df = pd.read_csv(r'.\三级实操题-202603-加密\telecom_churn_test.csv')

print(f"训练集大小: {train_df.shape}")
print(f"测试集大小: {test_df.shape}")

# 保存测试集的id用于最后提交
test_ids = test_df['id'].copy()

加载数据...
训练集大小: (389712, 22)
测试集大小: (97428, 21)


In [4]:
# threshold = 0.6
# train_df = train_df.dropna(thresh=int(len(train_df.columns) * threshold))

In [5]:
train_df['Churn'].value_counts(normalize=True)

No     0.945062
Yes    0.054938
Name: Churn, dtype: float64

In [6]:
train_df.columns

Index(['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'FavoriteColor', 'LuckyNumber', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges',
       'Churn'],
      dtype='object')

In [7]:
# 1.处理应为数值类型的变量，但原始数据可能包含空格或空字符串
# 先转换为字符串，去除空格，然后转为float，无效值变为NaN
def clean_numdata(df,cols):
    for col in cols:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        except:print('err : '+col)
    return df
# 2.处理数值型特征的缺失值,若有则用中位数填充
def fill_numdata(df,cols):
    for col in cols:
        try:
            df[col] = df[col].fillna(df[col].median())
        except:print('err : '+col)
    return df
# 3. 处理类别型特征的缺失值（用众数填充）,或用固定值填补
def fill_classdata(df,cols):
    for col in cols:
        if col in df.columns:
            # mode_val = df[col].mode()[0]
            # df[col].fillna(mode_val, inplace=True)
            df[col] = df[col].fillna('Unknown')
    return df

train_df = clean_numdata(train_df,cols=['LuckyNumber','MonthlyCharges','TotalCharges'])
train_df = fill_numdata(train_df,cols=['LuckyNumber','MonthlyCharges','TotalCharges'])
train_df = fill_classdata(train_df,cols=[ 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
                                         'FavoriteColor', 'LuckyNumber', 'PhoneService', 'MultipleLines',
                                         'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                                         'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
                                         'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges'])

test_df = clean_numdata(test_df,cols=['LuckyNumber','MonthlyCharges','TotalCharges'])
test_df = fill_numdata(test_df,cols=['LuckyNumber','MonthlyCharges','TotalCharges'])
test_df = fill_classdata(test_df,cols=[ 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
                                         'FavoriteColor', 'LuckyNumber', 'PhoneService', 'MultipleLines',
                                         'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                                         'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
                                         'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges'])

In [8]:
train_df = train_df.drop(['FavoriteColor','LuckyNumber'],axis=1)
test_df = test_df.drop(['FavoriteColor','LuckyNumber'],axis=1)

def MakeNewCol(data):
    # 构造新特征：使用时长 ≈ TotalCharges / MonthlyCharges （注意分母为零）
    data['tenure'] = 0.0
    mask = data['MonthlyCharges'] > 0
    data.loc[mask, 'tenure'] = data.loc[mask, 'TotalCharges'] / data.loc[mask, 'MonthlyCharges']
    return data
train_df = MakeNewCol(train_df)
test_df = MakeNewCol(test_df)

In [9]:
# 标签编码 Churn
churn_map = {'Yes': 1, 'No': 0}
train_df['Churn'] = train_df['Churn'].map(churn_map)

# 保存id
train_ids = train_df['id']
test_ids  = test_df['id']

In [10]:
# 4.3 分离特征和目标
y = train_df['Churn']
X = train_df.drop(columns=['id', 'Churn'])
X_test = test_df.drop(columns=['id'])

In [11]:
# 4.4 识别特征类型
cat_features = X.select_dtypes(include='object').columns.tolist()
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
print(f"\n  类别型特征 ({len(cat_features)}): {cat_features}")
print(f"  数值型特征 ({len(num_features)}): {num_features}")


  类别型特征 (16): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
  数值型特征 (3): ['MonthlyCharges', 'TotalCharges', 'tenure']


In [13]:
# 4.5 类别型特征编码
def encode_features(X_train, X_test, cat_cols):
    """对类别型特征进行编码"""
    encoders = {}
    X_train = X_train.copy()
    X_test  = X_test.copy()

    for col in cat_cols:
        # print(col)
        le = LabelEncoder()
        # 合并训练集和测试集的唯一值进行编码
        all_vals = set(X_train[col].dropna().unique()) | set(X_test[col].dropna().unique())
        le.fit(list(all_vals))

        X_train[col] = le.transform(X_train[col].astype(str))
        X_test[col]  = le.transform(X_test[col].astype(str))
        encoders[col] = le
        # print(f"  {col}: {len(le.classes_)} 个类别 -> 编码为 0~{len(le.classes_)-1}")

    return X_train, X_test, encoders

In [14]:
print("\n--- 类别型特征编码 ---")
X, X_test, encoders = encode_features(X, X_test, cat_features)


--- 类别型特征编码 ---


In [15]:
# 4.6 衍生特征工程
print("\n--- 衍生新特征 ---")

# 4.6.1 平均月消费
X['AvgMonthlyCharge'] = X['TotalCharges'] / (X['tenure'] + 1)
X_test['AvgMonthlyCharge'] = X_test['TotalCharges'] / (X_test['tenure'] + 1)
print("  新增特征: AvgMonthlyCharge (总消费/月数)")

# 4.6.2 月消费变化率
X['ChargeChange'] = X['MonthlyCharges'] - X['AvgMonthlyCharge']
X_test['ChargeChange'] = X_test['MonthlyCharges'] - X_test['AvgMonthlyCharge']
print("  新增特征: ChargeChange (当前月费 - 平均月费)")

# 4.6.3 总消费/月消费比率
X['TotalToMonthlyRatio'] = X['TotalCharges'] / (X['MonthlyCharges'] + 1e-6)
X_test['TotalToMonthlyRatio'] = X_test['TotalCharges'] / (X_test['MonthlyCharges'] + 1e-6)
print("  新增特征: TotalToMonthlyRatio")

# 4.6.4 是否新增客户 (tenure <= 6)
X['IsNewCustomer'] = (X['tenure'] <= 6).astype(int)
X_test['IsNewCustomer'] = (X_test['tenure'] <= 6).astype(int)
print("  新增特征: IsNewCustomer (tenure <= 6)")

# 4.6.5 是否长期客户 (tenure >= 48)
X['IsLongTermCustomer'] = (X['tenure'] >= 48).astype(int)
X_test['IsLongTermCustomer'] = (X_test['tenure'] >= 48).astype(int)
print("  新增特征: IsLongTermCustomer (tenure >= 48)")

# 4.6.6 附加服务数量
service_num_cols = []
for sc in ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
           'TechSupport', 'StreamingTV', 'StreamingMovies']:
    if sc in X.columns:
        service_num_cols.append(sc)

if service_num_cols:
    # 检查这些列是否已被编码 (Yes/No -> 1/0 或 LabelEncoder)
    # 先尝试按数值求和
    X['NumServices'] = X[service_num_cols].sum(axis=1)
    X_test['NumServices'] = X_test[service_num_cols].sum(axis=1)
    print(f"  新增特征: NumServices (附加服务数量, 基于 {service_num_cols})")



--- 衍生新特征 ---
  新增特征: AvgMonthlyCharge (总消费/月数)
  新增特征: ChargeChange (当前月费 - 平均月费)
  新增特征: TotalToMonthlyRatio
  新增特征: IsNewCustomer (tenure <= 6)
  新增特征: IsLongTermCustomer (tenure >= 48)
  新增特征: NumServices (附加服务数量, 基于 ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'])


In [17]:
# fig,ax = plt.subplots(figsize = (7,3))
# plt.hist(X['tenure'], bins=60, alpha=0.7, color='grey')
# plt.axvline(88, color='red', linestyle='--', lw=0.86,alpha=0.66)
# plt.ylabel('Frequency')
# plt.show()

In [18]:
# 4.6.7 tenure 分桶
X['TenureBin'] = pd.cut(X['tenure'], bins=[0, 12, 24, 48, 72,88, np.inf],
                         labels=[0, 1, 2, 3, 4,5], include_lowest=True).astype(int)
X_test['TenureBin'] = pd.cut(X_test['tenure'], bins=[0, 12, 24, 48, 72,88, np.inf],
                              labels=[0, 1, 2, 3, 4,5], include_lowest=True).astype(int)
print("  新增特征: TenureBin (tenure分段)")

  新增特征: TenureBin (tenure分段)


In [19]:
# # 4.7 特征重要性初探（使用随机森林快速评估）
# print("\n--- 特征重要性 (RandomForest) ---")
# rf_temp = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
# rf_temp.fit(X, y)

# feat_importance = pd.DataFrame({
#     'feature': X.columns,
#     'importance': rf_temp.feature_importances_
# }).sort_values('importance', ascending=False)

# plt.figure(figsize=(10, 8))
# sns.barplot(data=feat_importance.head(20), x='importance', y='feature')
# plt.title('Top 20 特征重要性')
# plt.tight_layout()
# plt.savefig('feature_importance.png', dpi=150)
# plt.show()

# print(feat_importance.to_string(index=False))


In [20]:
# ==================== 5. 模型训练与评估 ====================
print("\n" + "=" * 60)
print("Step 5: 模型训练与评估")
print("=" * 60)


Step 5: 模型训练与评估


In [21]:
# 5.1 划分验证集
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"  训练集: {X_train.shape[0]}, 验证集: {X_val.shape[0]}")
print(f"  训练集 Churn 比例: {y_train.mean():.4f}")
print(f"  验证集 Churn 比例: {y_val.mean():.4f}")

# 5.2 计算类别权重（处理不平衡）
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f"  正负样本比: {pos_count}:{neg_count}")
print(f"  scale_pos_weight: {scale_pos_weight:.4f}")

  训练集: 311769, 验证集: 77943
  训练集 Churn 比例: 0.0549
  验证集 Churn 比例: 0.0549
  正负样本比: 17128:294641
  scale_pos_weight: 17.2023


In [51]:
# 5.3 定义多个模型
models = {
    'LightGBM': LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        use_label_encoder=False,
        verbosity=0
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=500,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=4,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5,
        subsample=0.8,
        random_state=42
    )
}

# 5.4 训练并评估每个模型
best_auc = 0
best_model_name = None

for name, model in models.items():
    print(f"\n--- 训练 {name} ---")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]
    
    auc_0 = roc_auc_score(y_val, y_prob)
    f1_0  = f1_score(y_val, y_pred)
    accuracy_0 = accuracy_score(y_val, y_pred)
    print(f"  验证集 AUC_0: {auc_0:.5f}, F1_0: {f1_0:.5f}, accuracy_score_0: {accuracy_0:.5f}")
    
    # 基于 PR 曲线找最优（更稳定）
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_prob)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_thresh_pr = thresholds[best_idx]
    best_f1_pr = f1_scores[best_idx]
    # print(f"PR曲线最优阈值: {best_thresh_pr:.3f}, F1: {best_f1_pr:.5f}")
    # 应用最优阈值
    y_pred = (y_prob >= best_thresh_pr).astype(int)

    f1_1  = f1_score(y_val, y_pred)
    accuracy_1 = accuracy_score(y_val, y_pred)
    print(f"  调整阈值后  F1_1: {f1_1:.5f}, accuracy_score_1: {accuracy_1:.5f}")

    if auc_0 > best_auc:
        best_auc = auc_0
        best_model_name = name

print(f"\n{'='*40}")
print(f"最佳单模型: {best_model_name} (AUC={best_auc:.5f})")
print(f"{'='*40}")




--- 训练 LightGBM ---
  验证集 AUC_0: 0.89701, F1_0: 0.33694, accuracy_score_0: 0.83034
  调整阈值后  F1_1: 0.40275, accuracy_score_1: 0.91431

--- 训练 XGBoost ---
  验证集 AUC_0: 0.89936, F1_0: 0.32879, accuracy_score_0: 0.81498
  调整阈值后  F1_1: 0.40833, accuracy_score_1: 0.92456

--- 训练 RandomForest ---
  验证集 AUC_0: 0.89437, F1_0: 0.35880, accuracy_score_0: 0.85849
  调整阈值后  F1_1: 0.39023, accuracy_score_1: 0.92181

--- 训练 GradientBoosting ---
  验证集 AUC_0: 0.89793, F1_0: 0.18575, accuracy_score_0: 0.94691
  调整阈值后  F1_1: 0.39904, accuracy_score_1: 0.91286

最佳单模型: XGBoost (AUC=0.89936)


In [ ]:
先保存一份结果，以防调优时间不够

In [70]:
model =  LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

model.fit(X, y)

y_pred = model.predict(X)
y_prob = model.predict_proba(X)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y, y_prob)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_thresh_pr = thresholds[best_idx]


In [76]:
y_test = model.predict_proba(X_test)[:, 1]
y_test = (y_test >= best_thresh_pr).astype(int)

In [79]:
y_test

array([0, 0, 0, ..., 0, 0, 0])

In [81]:
submission_opt = pd.DataFrame({
    'id': test_ids,
    'Churn': y_test
})

In [83]:
submission_opt['Churn'].value_counts()

0    90910
1     6518
Name: Churn, dtype: int64

In [ ]:
有时间的话，先用最快的lightGBM调参看看。。。。

In [60]:
import optuna
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np

In [66]:
# optuna.logging.set_verbosity(optuna.logging.WARNING) # 关闭冗余日志

# def objective(trial):
#     # 1. 定义搜索空间
#     param = {
#         'objective': 'binary',
#         'metric': 'binary_logloss',
#         'boosting_type': 'gbdt',
#         'seed': 42,
#         'n_jobs': -1,
#         'verbose': -1,
        
#         # Optuna 自动在这里采样
#         'num_leaves': trial.suggest_int('num_leaves', 15, 127),
#         'max_depth': trial.suggest_int('max_depth', -1, 10),
#         'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
#         'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
#         'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
#         'scale_pos_weight': (y == 0).sum() / (y == 1).sum()
#     }
    
#     model = LGBMClassifier(**param)
#     # 使用交叉验证
#     scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
#     return scores.mean()
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=20)

# # # 输出结果
# # print("\n" + "="*50)
# # print("调参完成！")
# # print(f"最佳 F1 得分: {study.best_value:.5f}")
# # print(f"最佳参数: ")
# # for key, value in study.best_params.items():
# #     print(f"    {key}: {value}")

In [ ]:
"""若有时间，可尝试模型融合。。。。。。。"""

In [ ]:
# ==================== 6. 模型融合（Stacking集成） ====================
print("\n" + "=" * 60)
print("Step 6: 模型融合 (Stacking)")
print("=" * 60)

# 6.1 使用 K-Fold 生成 OOF 预测
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# 选择表现较好的几个模型进行融合
stack_models = {
    'LightGBM': LGBMClassifier(
        n_estimators=1000, learning_rate=0.05, num_leaves=63,
        max_depth=-1, min_child_samples=20, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
        scale_pos_weight=scale_pos_weight, random_state=42,
        n_jobs=-1, verbose=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=1000, learning_rate=0.05, max_depth=6,
        min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=scale_pos_weight,
        random_state=42, n_jobs=-1, eval_metric='logloss',
        use_label_encoder=False, verbosity=0
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=500, max_depth=15, min_samples_split=10,
        min_samples_leaf=4, class_weight='balanced',
        random_state=42, n_jobs=-1
    )
}

oof_preds = np.zeros((len(X), len(stack_models)))
test_preds = np.zeros((len(X_test), len(stack_models)))

for i, (name, model) in enumerate(stack_models.items()):
    print(f"\n--- Stacking: 训练 {name} (5-Fold CV) ---")
    oof_pred = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

        clone_model = __import__('sklearn.base', fromlist=['clone']).clone(model)
        clone_model.fit(X_tr, y_tr)

        oof_pred[val_idx] = clone_model.predict_proba(X_va)[:, 1]
        test_pred += clone_model.predict_proba(X_test)[:, 1] / n_folds

        fold_auc = roc_auc_score(y_va, oof_pred[val_idx])
        print(f"    Fold {fold+1}: AUC = {fold_auc:.5f}")

    oof_auc = roc_auc_score(y, oof_pred)
    print(f"  {name} OOF AUC: {oof_auc:.5f}")

    oof_preds[:, i] = oof_pred
    test_preds[:, i] = test_pred

# 6.2 用简单加权平均融合
# 权重可以根据OOF AUC自动确定
weights = []
for i, (name, model) in enumerate(stack_models.items()):
    oof_auc = roc_auc_score(y, oof_preds[:, i])
    weights.append(oof_auc)

# 归一化权重
weights = np.array(weights)
weights = weights / weights.sum()
print(f"\n--- 融合权重 ---")
for i, name in enumerate(stack_models.keys()):
    print(f"  {name}: {weights[i]:.4f}")

# 加权平均
final_oof = np.average(oof_preds, axis=1, weights=weights)
final_test = np.average(test_preds, axis=1, weights=weights)

final_oof_auc = roc_auc_score(y, final_oof)
print(f"\n融合后 OOF AUC: {final_oof_auc:.5f}")

# 6.3 尝试将OOF预测作为元特征，训练二层模型 (Logistic Regression)
from sklearn.linear_model import LogisticRegression as LR

meta_model = LR(C=1.0, max_iter=1000, random_state=42)
meta_model.fit(oof_preds, y)
meta_oof = meta_model.predict_proba(oof_preds)[:, 1]
meta_test = meta_model.predict_proba(test_preds)[:, 1]

meta_auc = roc_auc_score(y, meta_oof)
print(f"Stacking (LR二层) OOF AUC: {meta_auc:.5f}")
print(f"  二层模型系数: {dict(zip(stack_models.keys(), meta_model.coef_[0]))}")

# 选择更优的融合策略
if meta_auc > final_oof_auc:
    final_test_pred = meta_test
    print(f"\n✓ 最终采用 Stacking (LR二层) 融合, AUC={meta_auc:.5f}")
else:
    final_test_pred = final_test
    print(f"\n✓ 最终采用 加权平均 融合, AUC={final_oof_auc:.5f}")


# ==================== 7. 最终训练与预测 ====================
print("\n" + "=" * 60)
print("Step 7: 最终训练与提交")
print("=" * 60)

# 使用全量训练数据重新训练最佳模型
print("使用全量数据重新训练最佳模型...")

best_model = None
if meta_auc >= final_oof_auc:
    # Stacking方案：全量训练base模型，再训练meta
    print("采用 Stacking 方案...")
    for i, (name, model) in enumerate(stack_models.items()):
        model.fit(X, y)
        test_preds[:, i] = model.predict_proba(X_test)[:, 1]
        print(f"  {name} 训练完成")

    meta_model.fit(test_preds, y)  # 注意: 这里用训练数据的预测
    # 但更合理的做法是用 OOF 来训练 meta
    # 已经在 Step 6 中训练好了 meta_model，直接使用
    final_predictions_prob = meta_test
else:
    # 加权平均方案
    print("采用加权平均方案...")
    for i, (name, model) in enumerate(stack_models.items()):
        model.fit(X, y)
        test_preds[:, i] = model.predict_proba(X_test)[:, 1]
        print(f"  {name} 训练完成")
    final_predictions_prob = np.average(test_preds, axis=1, weights=weights)

# 转换为二分类预测 (0/1)
final_predictions = (final_predictions_prob >= 0.5).astype(int)

# 生成提交文件
submission = pd.DataFrame({
    'id': test_ids,
    'Churn': final_predictions
})
submission.to_csv('submission.csv', index=False)
print(f"\n提交文件已保存: submission.csv")
print(f"预测分布: {pd.Series(final_predictions).value_counts().to_dict()}")

# 同时保存概率预测（可用于调整阈值优化F1）
submission_prob = pd.DataFrame({
    'id': test_ids,
    'Churn_Prob': final_predictions_prob
})
submission_prob.to_csv('submission_prob.csv', index=False)
print(f"概率文件已保存: submission_prob.csv")


# ==================== 8. 阈值优化 ====================
print("\n" + "=" * 60)
print("Step 8: 阈值优化")
print("=" * 60)

# 在OOF预测上搜索最优阈值
best_threshold = 0.5
best_f1_score = 0

for threshold in np.arange(0.2, 0.8, 0.01):
    pred = (final_oof >= threshold).astype(int)
    f1 = f1_score(y, pred)
    if f1 > best_f1_score:
        best_f1_score = f1
        best_threshold = threshold

print(f"最优阈值: {best_threshold:.2f}, 对应F1: {best_f1_score:.5f}")

# 用最优阈值重新生成提交
final_predictions_optimized = (final_predictions_prob >= best_threshold).astype(int)
submission_opt = pd.DataFrame({
    'id': test_ids,
    'Churn': final_predictions_optimized
})
submission_opt.to_csv('submission_optimized.csv', index=False)
print(f"优化后提交文件已保存: submission_optimized.csv")
print(f"优化后预测分布: {pd.Series(final_predictions_optimized).value_counts().to_dict()}")

print("\n" + "=" * 60)
print("全部完成！")
print("=" * 60)
print(f"""
生成文件清单:
  1. churn_distribution.png  - Churn分布图
  2. numerical_features.png   - 数值特征分布图
  3. correlation_heatmap.png  - 相关性热力图
  4. feature_importance.png   - 特征重要性图
  5. model_comparison.png     - 模型对比图
  6. submission.csv           - 基础提交 (阈值=0.5)
  7. submission_prob.csv      - 概率预测
  8. submission_optimized.csv - 阈值优化提交
""")